<h1 style="font-family: 'Times New Roman'; text-align: center;"><b>Machine Learning</b></h1>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b>URL Phishing Detection Project</b></h2>

<h3 style="font-family: 'Times New Roman'; text-align: right;"><b>- Pallav Biyala (24110234)</b></h3>

<HR>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b> Pre-Processing of the Data</b></h2>

### Importing Libraries

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
df = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv")
df1 = df.copy() # Creating a copy if required to use in future

### Forming Features Vector and Target Vector

In [5]:
#### Removing the Duplicated URLs we found in the EDA
print(f"Original shape: {df.shape}")

df = df.drop_duplicates(subset=['URL'], keep='first').copy()

print(f"Cleaned shape:  {df.shape}")
print(f"Rows removed:   {len(df1) - len(df)}")
print("Labels after Removing Duplication")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize = True))

Original shape: (235795, 56)
Cleaned shape:  (235370, 56)
Rows removed:   425
Labels after Removing Duplication
label
1    134850
0    100520
Name: count, dtype: int64
label
1    0.572928
0    0.427072
Name: proportion, dtype: float64


In [21]:
### So in features, we have 50 features excluding label and rest 5 string valued features
X = df.drop(columns=["label", "FILENAME", "URL", "Domain", "TLD", "Title"], errors="ignore")
print("X shape: ",X.shape )

X shape:  (235370, 50)


In [22]:
### And in y we have only lavel
y = df["label"]
print("y shape: ", y.shape)

y shape:  (235370,)


### Splitting the Datase

<h3 style="font-family: 'Times New Roman';"> So firstly, we should divide our dataset into three parts:</h3>
<ol style="font-family: 'Times New Roman';font-size: 20px">
    <li>Training Set</li>
    <li>Validation Set</li>
    <li>Testing Set</li>
</ol>
<h3 style="font-family: 'Times New Roman';">Now, as we saw in the EDA stage, our data has nearly 57% of Legitimate URLs and rest phishing URLs. Now it is very important that while splitting our database, we don't affect this ratio. Otherwise it is possible that during training our model gets majorly legitimate and then it couldn't learn how to distinguish with phishing ones. Hence, to maintain this ratio, we use <strong>stratification</strong></h3>
<h3 style="font-family: 'Times New Roman';">Stratification helps us to prevent uneven class divsions among the three sets thereby maintaining the original ratio.</h3>
<h3 style="font-family: 'Times New Roman';">So, let's divide it in 80:10:10 ratio (train:validate:test)</h3>

In [23]:
# Dividng dataset into (Train+Val and test)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,y,
    test_size=0.1,
    random_state=42,
    stratify=y # making sure we have similar ratio across both classes across all two sets
)

In [24]:
# now, we can split the remaining 90% of train + val ito 80% train and 10% val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=1/9,
    random_state=42,
    stratify=y_train_val
)

In [25]:
### Checking the dataset split
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)
print("Total entries:", len(y_train) + len(y_val) + len(y_test))

Training: (188296, 50) (188296,)
Validation: (23537, 50) (23537,)
Testing: (23537, 50) (23537,)
Total entries: 235370


In [26]:
### Now, we can also check the dataset distribution:
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))

Training distribution:
label
1    0.572928
0    0.427072
Name: proportion, dtype: float64

Validation distribution:
label
1    0.572928
0    0.427072
Name: proportion, dtype: float64

Testing distribution:
label
1    0.572928
0    0.427072
Name: proportion, dtype: float64


### Feature Scaling

#### During EDA, we observed that our numerical features have very different ranges. For example, `URLLength` can reach thousands, while some features are binary. Such differences in scale can affect the optimization and regularization of certain machine learning algorithms. Therefore, we use StandardScaler to standardize the features by subtracting their training-set mean and dividing by their training-set standard deviation.

#### We fit the scaler only on the training set and use the learned statistics to transform the training, validation, and test sets. This prevents information from the validation and test sets from influencing the scaling process.

#### So to scale our features, we use:

<center> $z = \frac{x - \mu}{\sigma}$ </center>

 #### So, z becomes our scaled features. So now to do this, we use: fit() and transform()

In [27]:
# so now we find the statistics of training set and make our model learn them
scaler = StandardScaler()
scaler.fit(X_train)

# Now we apply transformation
X_train_scaled = scaler.transform(X_train)

# Using validation or test data to fit the scaler would leak information from those datasets into preprocessing. Thats why we scale using X_train only.
# So thats why the lines: .fit() and .transform() for X_train can be also written as: X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Testing:", X_test_scaled.shape)

Training: (188296, 50)
Validation: (23537, 50)
Testing: (23537, 50)


In [28]:
## Let's check if it worked
print("Training feature means (first 5):")
print(X_train_scaled.mean(axis=0)[:5])

print("\nTraining feature standard deviations (first 5):")
print(X_train_scaled.std(axis=0)[:5])

Training feature means (first 5):
[-4.10561295e-17  8.67914503e-17  9.50932412e-18  5.10258257e-16
 -3.17883121e-16]

Training feature standard deviations (first 5):
[1. 1. 1. 1. 1.]


<hr>